# 02 - Demand Modeling: Non-Homogeneous Poisson Process (NHPP)

This notebook implements and validates the NHPP demand model using the thinning (Lewis-Shedler) algorithm. It estimates arrival rates by precinct, hour-of-day, and day-of-week.

**Runtime:** ~5 minutes  
**Data Required:** Processed lambda tables, crashes data

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally")
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

## Load Lambda Tables

In [ ]:
hourly = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_hourly.csv'))
dow = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_dow.csv'))
precinct = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv'))

print("=== HOURLY LAMBDA FACTORS ===")
display(hourly)
print(f"\n=== DAY-OF-WEEK LAMBDA FACTORS ===")
display(dow)
print(f"\n=== PRECINCT LAMBDA RATES (top 10) ===")
display(precinct.head(10))
print(f"\nTotal precincts: {len(precinct)}")
print(f"Base rate: {hourly['lambda_per_hour'].mean():.4f} crashes/hour")

## Visualize Lambda Profiles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hourly
axes[0].plot(hourly['hour'], hourly['lambda_per_hour'], 'b-o', markersize=4)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Lambda (crashes/hour)')
axes[0].set_title('Hourly Arrival Rate')
axes[0].axhline(y=hourly['lambda_per_hour'].mean(), color='red', linestyle='--', alpha=0.7)

# Day of week
axes[1].bar(dow['day_name'], dow['factor'], color='darkorange')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Factor (relative to mean)')
axes[1].set_title('Day-of-Week Factor')
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.7)
axes[1].tick_params(axis='x', rotation=45)

# Precinct
precinct_sorted = precinct.sort_values('lambda_per_hour', ascending=True)
axes[2].barh(precinct_sorted['precinct'].astype(str), precinct_sorted['lambda_per_hour'], color='seagreen')
axes[2].set_xlabel('Lambda (crashes/hour)')
axes[2].set_ylabel('Precinct')
axes[2].set_title('Precinct Arrival Rates')

plt.tight_layout()
save_output(fig, 'lambda_profiles.png', 'figures/demand')
plt.show()

## NHPP Arrival Generator

The NHPP uses the thinning algorithm to generate non-homogeneous arrivals:
1. Compute envelope rate `lambda_max`
2. Generate homogeneous Poisson with rate `lambda_max`
3. Thin (accept/reject) based on ratio `lambda(t) / lambda_max`

In [ ]:
from ems_readiness.demand.arrival_generator import NHPPArrivalGenerator

# Initialize from tables
gen = NHPPArrivalGenerator.from_tables(data_dir=PROCESSED_DIR)
print(f"Base rate: {gen.base_rate:.4f} crashes/hour")
print(f"Hourly factors loaded: {len(gen.hourly_factors)} hours")
print(f"DOW factors loaded: {len(gen.dow_factors)} days")
if gen.precinct_probs is not None:
    print(f"Precinct probabilities: {len(gen.precinct_probs)} precincts")

## Generate Sample Arrivals

In [ ]:
# Generate arrivals for one week (Monday start)
np.random.seed(42)
week_arrivals = []
for day in range(7):
    day_arrivals = gen.generate_arrivals(n_hours=24, start_hour=0, dow=day, rng=42+day)
    day_arrivals['day'] = day
    day_arrivals['absolute_time'] = day_arrivals['time_hours'] + day * 24
    week_arrivals.append(day_arrivals)

all_arrivals = pd.concat(week_arrivals, ignore_index=True)
print(f"Total arrivals in simulated week: {len(all_arrivals):,}")
print(f"Expected: ~{3.48 * 168:.0f} (base_rate * 168 hours)")
print(f"Ratio: {len(all_arrivals) / (3.48 * 168):.3f}")
display(all_arrivals.head(10))

## Validate Arrival Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histogram of inter-arrival times
inter_arrivals = all_arrivals['absolute_time'].diff().dropna() * 60  # minutes
axes[0, 0].hist(inter_arrivals, bins=50, density=True, color='steelblue', edgecolor='white', alpha=0.7)
axes[0, 0].set_xlabel('Inter-arrival Time (minutes)')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Inter-arrival Time Distribution')

# Hourly counts
hourly_sim = all_arrivals.groupby('hour').size()
hourly_emp = hourly['lambda_per_hour'] * 7  # expected over a week
axes[0, 1].bar(hourly_sim.index - 0.2, hourly_sim.values, 0.4, label='Simulated', color='steelblue')
axes[0, 1].bar(hourly_emp.index if hasattr(hourly_emp, 'index') else range(24),
               hourly['lambda_per_hour'].values * 7, 0.4, label='Expected', color='coral', alpha=0.7)
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Arrivals')
axes[0, 1].set_title('Hourly Arrivals: Simulated vs Expected')
axes[0, 1].legend()

# Arrivals per day
daily_counts = all_arrivals.groupby('day').size()
axes[1, 0].bar(range(7), daily_counts.values, color='darkorange')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1, 0].set_ylabel('Arrivals')
axes[1, 0].set_title('Daily Arrivals (Simulated Week)')

# Precinct distribution
if 'precinct' in all_arrivals.columns:
    prec_counts = all_arrivals.groupby('precinct').size().sort_values(ascending=False)
    axes[1, 1].barh(prec_counts.index.astype(str)[:15], prec_counts.values[:15], color='seagreen')
    axes[1, 1].set_xlabel('Arrivals')
    axes[1, 1].set_ylabel('Precinct')
    axes[1, 1].set_title('Top 15 Precincts by Arrivals')

plt.tight_layout()
save_output(fig, 'nhpp_validation.png', 'figures/demand')
plt.show()

## CBD vs Non-CBD Demand Factors

In [ ]:
if 'cbd_factor' in hourly.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hourly['hour'], hourly['cbd_factor'], 'b-o', label='CBD', markersize=4)
    axes[0].plot(hourly['hour'], hourly['non_cbd_factor'], 'r-s', label='Non-CBD', markersize=4)
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('Factor')
    axes[0].set_title('Hourly Factors: CBD vs Non-CBD')
    axes[0].legend()
    axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

    if 'cbd_factor' in dow.columns:
        x = range(7)
        w = 0.35
        axes[1].bar([i-w/2 for i in x], dow['cbd_factor'], w, label='CBD', color='steelblue')
        axes[1].bar([i+w/2 for i in x], dow['non_cbd_factor'], w, label='Non-CBD', color='coral')
        axes[1].set_xticks(list(x))
        axes[1].set_xticklabels(dow['day_name'], rotation=45)
        axes[1].set_ylabel('Factor')
        axes[1].set_title('DOW Factors: CBD vs Non-CBD')
        axes[1].legend()
        axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    save_output(fig, 'cbd_demand_factors.png', 'figures/demand')
    plt.show()

## Summary

- NHPP model captures time-varying arrival rates via thinning algorithm
- Hourly factors range from ~0.6 (nighttime) to ~1.4 (afternoon peak)
- Day-of-week factors show Friday peak, weekend reduction
- 30 precincts with heterogeneous demand rates
- CBD precincts account for ~56% of total demand